In [18]:
import os, glob, re, unicodedata, joblib
import numpy as np
import pandas as pd

# ================= CONFIG =================
DATA_DIR = "Dataset"
MODEL_PATH = "artifacts/ids_ensemble.joblib"

# ================= UTIL =================
def normalize_colname(name):
    s = str(name).strip()
    s = s.replace("%", "pct")
    s = re.sub(r"[\/\\\s\-\.\(\)]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_").lower()
    return s

def clean_label(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s)
    return s.strip().upper()

def safe_numeric(df):
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.replace([np.inf, -np.inf], np.nan).fillna(0)

# ================= LOAD MODEL =================
bundle = joblib.load(MODEL_PATH)
model_bin = bundle["bin"]      # Stage A
model_mul = bundle["mul"]      # Stage B
features  = bundle["features"]
THR       = bundle["best_threshold"]

print("Loaded model")
print("Stage-A threshold:", THR)

# ================= LOAD DATASET =================
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
dfs = []

for f in files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = [normalize_colname(c) for c in df.columns]
    df["capture_id"] = os.path.basename(f)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

# Keep original labels if present (for analysis only)
label_col = next((c for c in df.columns if "label" in c), None)
if label_col:
    df["true_label"] = df[label_col].map(clean_label)

# ================= PREPARE FEATURES =================
X = df.reindex(columns=features, fill_value=0)
X = safe_numeric(X)

# ================= STAGE A: ATTACK DETECTION =================
df["prob_attack"] = model_bin.predict_proba(X)[:, 1]
df["is_attack"] = df["prob_attack"] >= THR

# ================= FILTER ONLY ATTACKS =================
attacks = df[df["is_attack"]].copy()

print(f"\nTotal flows        : {len(df):,}")
print(f"Predicted attacks  : {len(attacks):,}")

# ================= STAGE B: ATTACK FAMILY =================
if len(attacks) > 0:
    X_atk = attacks.reindex(columns=features, fill_value=0)
    X_atk = safe_numeric(X_atk)

    attacks["predicted_family"] = model_mul.predict(X_atk)

    if hasattr(model_mul, "predict_proba"):
        fam_proba = model_mul.predict_proba(X_atk)
        attacks["family_confidence"] = fam_proba.max(axis=1)
else:
    attacks["predicted_family"] = []
    attacks["family_confidence"] = []

# ================= OUTPUT =================
cols_to_show = [
    "capture_id",
    "prob_attack",
    "predicted_family",
    "family_confidence"
]

if "true_label" in attacks.columns:
    cols_to_show.insert(2, "true_label")

print("\nSample predicted attacks:")
print(attacks[cols_to_show].head(20))

# ================= OPTIONAL: SAVE =================
attacks.to_csv("predicted_attacks_only.csv", index=False)
print("\nSaved → predicted_attacks_only.csv")


Loaded model
Stage-A threshold: 0.999859849387428

Total flows        : 2,830,743
Predicted attacks  : 0

Sample predicted attacks:
Empty DataFrame
Columns: [capture_id, prob_attack, true_label, predicted_family, family_confidence]
Index: []

Saved → predicted_attacks_only.csv


In [19]:
print("Max prob_attack:", df["prob_attack"].max())
print("Mean prob_attack:", df["prob_attack"].mean())
print("Threshold:", THR)


Max prob_attack: 0.9190454358910632
Mean prob_attack: 0.0016213090806939576
Threshold: 0.999859849387428
